# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Signal 1 verdict: MIXED — staleness bucket vs decline rate.
<90d: 0.512 (n=20,655), 90-180d: 0.611 (n=9,171), 180-365d: 0.467 (n=169), 365d+: 0.600 (n=5).
Not a clean trend, decline rate actually goes UP then DOWN then UP again as staleness
increases, and the two oldest buckets have too few pages (169 and 5) to trust. This matches
what we already suspected from week 1: staleness isn't a clean predictor of decline on its
own. Calling this MIXED rather than FALSE only because 90-180d does show a real bump, but it's not the clean "older = more declining" story a refresh-flag rule assumes.

Signal 2 verdict: MIXED — impressions_90d (demand) vs decline rate.
Bottom quartile: 0.376, then 0.605, 0.626, and top quartile drops back to 0.562 (n≈7,500
each, solid sample size). Demand does separate low-impression pages (less likely declining)
from everything else, but it's not monotonic; the very highest-demand pages decline LESS
than mid-demand pages, not more. So "more demand = more worth reviewing" holds directionally
in the bottom half, but breaks at the top. Worth remembering when combining with staleness.

Rule in plain words: "A page is worth reviewing first if it's currently declining and still
gets meaningful search demand (>=100 impressions/90d) staleness alone doesn't cleanly
predict decline, so I'm not using it in the rule itself, only checking it as a candidate
signal."

In [7]:
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Signal 1: staleness (behind FlyRank's refresh flags)
df['stale_bucket'] = pd.cut(df['days_since_last_update'], bins=[-1,90,180,365,10000],
                             labels=['<90d','90-180d','180-365d','365d+'])
stale_check = df.groupby('stale_bucket', observed=True).agg(
    n=('trend_direction','size'),
    decline_rate=('trend_direction', lambda x: (x=='down').mean())
)
print("SIGNAL 1: staleness (days_since_last_update) vs decline rate")
print(stale_check)

# Signal 2: demand/volume (behind quick-win logic)
df['impression_bucket'] = pd.qcut(df['impressions_90d'].clip(lower=0),
                                   q=[0,.25,.5,.75,1.0], duplicates='drop')
volume_check = df.groupby('impression_bucket', observed=True).agg(
    n=('trend_direction','size'),
    decline_rate=('trend_direction', lambda x: (x=='down').mean())
)
print("SIGNAL 2: impressions_90d (demand/volume) vs decline rate")
print(volume_check)

SIGNAL 1: staleness (days_since_last_update) vs decline rate
                  n  decline_rate
stale_bucket                     
<90d          20655      0.512031
90-180d        9171      0.611057
180-365d        169      0.467456
365d+             5      0.600000
SIGNAL 2: impressions_90d (demand/volume) vs decline rate
                        n  decline_rate
impression_bucket                      
(0.999, 81.0]        7503      0.376116
(81.0, 731.0]        7499      0.604614
(731.0, 3615.25]     7498      0.625634
(3615.25, 517715.0]  7500      0.562000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
declining = (df['trend_direction'] == 'down').astype(int)
has_demand = (df['impressions_90d'] >= 100).astype(int)

df['baseline_score'] = declining * has_demand * df['impressions_90d']
df['reason_code'] = 'declining_with_demand'
df['action'] = df['baseline_score'].apply(lambda s: 'review_for_refresh' if s > 0 else 'monitor')

print(df[['content_id','baseline_score','reason_code','action']].sort_values('baseline_score', ascending=False).head())

import numpy as np
import os

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

labels = (df['trend_direction'] == 'down').astype(int).values
base_rate = labels.mean()
p50 = precision_at_k(df['baseline_score'].values, labels, 50)

print(f"Base rate (random guessing): {base_rate:.3f}")
print(f"Precision@50: {p50:.3f}")

os.makedirs('work/outputs', exist_ok=True)
ranked = df.sort_values('baseline_score', ascending=False)
ranked[['content_id','client_id','baseline_score','reason_code','action']].to_csv(
    'work/outputs/baseline_action_score.csv', index=False)
print("Written to work/outputs/baseline_action_score.csv")

# save the metrics receipt (this one gets committed, unlike the CSV)
import json
with open('work/outputs/w04_metrics.json','w') as f:
    json.dump({'base_rate': float(base_rate), 'precision_at_50': float(p50)}, f, indent=2)

                 content_id  baseline_score            reason_code  \
6653   content_5fe46e04994d          517715  declining_with_demand   
26844  content_8c19996aa890          509252  declining_with_demand   
21819  content_4c36c775b818          463103  declining_with_demand   
29879  content_1a9e894be2e2          416180  declining_with_demand   
13537  content_2c2606c5d176          347399  declining_with_demand   

                   action  
6653   review_for_refresh  
26844  review_for_refresh  
21819  review_for_refresh  
29879  review_for_refresh  
13537  review_for_refresh  
Base rate (random guessing): 0.542
Precision@50: 1.000
Written to work/outputs/baseline_action_score.csv


## 3. Top-20 review

Top-10 review (all currently flagged "review_for_refresh", reason_code: declining_with_demand):

1. content_5fe46e04994d - score 517,715: highest-impression declining page in the dataset.
   Would be wrong if this is a seasonal top-of-funnel page where the "decline" is a normal
   yearly dip, not a real problem - worth checking trend_pct magnitude and content_type before
   trusting it blindly.
2. content_8c19996aa890 - score 509,252: same pattern, very high demand + declining.
   Would be wrong if a sibling/related page absorbed this traffic (consolidation, not decline)
   -- the lane guide warns about exactly this look-alike.
[... continue for all 10 once you paste the real rows]

In [9]:
top10 = ranked.head(10)[['content_id','baseline_score','reason_code','action','impressions_90d',
                          'days_since_last_update','trend_direction']]
top10

,content_id,baseline_score,reason_code,action,impressions_90d,days_since_last_update,trend_direction
6653,content_5fe46e04994d,517715,declining_with_demand,review_for_refresh,517715,104,down
26844,content_8c19996aa890,509252,declining_with_demand,review_for_refresh,509252,20,down
21819,content_4c36c775b818,463103,declining_with_demand,review_for_refresh,463103,20,down
29879,content_1a9e894be2e2,416180,declining_with_demand,review_for_refresh,416180,22,down
13537,content_2c2606c5d176,347399,declining_with_demand,review_for_refresh,347399,104,down
26531,content_cb112fce36be,309910,declining_with_demand,review_for_refresh,309910,104,down
21565,content_9532f197bbc8,309192,declining_with_demand,review_for_refresh,309192,104,down
27478,content_008fb02c46cb,236803,declining_with_demand,review_for_refresh,236803,20,down
23767,content_813e88069237,233561,declining_with_demand,review_for_refresh,233561,104,down
26304,content_ff94c9b6b411,228566,declining_with_demand,review_for_refresh,228566,20,down


## 4. Weak picks + leakage check

Weak-pick note: Precision@50 = 1.000 looks perfect, but it's an artifact, not a real result- baseline_score is built as declining * has_demand * impressions, so any nonzero score is
guaranteed to satisfy the "declining" label by construction. This isn't evidence the rule is
powerful; it's evidence the rule and the label are circularly tied together in this baseline
version. A more honest evaluation would score something NOT derived from the label itself
(e.g. rank purely by impressions among currently-declining pages, or evaluate against a
future-window label instead of the same-window trend_direction)- that's a real limitation
to carry into the model weeks, not swept under the rug.

In [10]:
# Confirm no label-derived or future-window columns snuck into the score
used_cols = ['trend_direction', 'impressions_90d']
print(f"Columns used in baseline_score: {used_cols}")
print("trend_direction is the LABEL SOURCE, used only to build the score being evaluated against itself here as a sanity check -- not as a leaked future feature, since this is the ranking target itself, not a hidden future outcome.")
print(f"No product-decision flags (health_score, priority_score, action_type) exist in this starter CSV, so none could leak in.")

Columns used in baseline_score: ['trend_direction', 'impressions_90d']
trend_direction is the LABEL SOURCE, used only to build the score being evaluated against itself here as a sanity check -- not as a leaked future feature, since this is the ranking target itself, not a hidden future outcome.
No product-decision flags (health_score, priority_score, action_type) exist in this starter CSV, so none could leak in.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.